# 2. Random Forest

Representation screening -> 5-fold grid search on the training split -> imbalance
study -> refit on train+val -> one test evaluation.

Every stage 03-05 notebook is independent; each one is a full experiment on its own
model family, and all of them write into the same `artifacts/` tree.


In [ ]:
# %% [markdown]
# Setup - dependencies and the project source
#
# This notebook runs the real pipeline from `src/`; it does not reimplement it.
# The cell below copies the project into the working directory so the stage can
# be imported and run unchanged.

import os, sys, subprocess, shutil, textwrap, zipfile, urllib.request
from pathlib import Path

# Kaggle ships most of these already. `datasets` is the one that is often missing.
QUIET = "-q"
subprocess.run([sys.executable, "-m", "pip", "install", QUIET,
                "datasets", "lightgbm", "imbalanced-learn", "joblib", "pyarrow"],
               check=False)

HERE = Path.cwd()          # a notebook has no __file__, so the cwd is the anchor
WORK = HERE / "enzyme_classification"
REPO_URL = os.environ.get("ENZYME_REPO_URL", "").strip()

def fetch_project(dest: Path) -> Path:
    """Materialise the project at `dest`.

    Resolution order:
      1. an already-extracted copy next to the notebook
      2. a zip uploaded as a Kaggle dataset / Colab file in /kaggle/input or /content
      3. a git clone from ENZYME_REPO_URL
    """
    if (dest / "src" / "config.py").exists():
        print(f"project already present at {dest}")
        return dest
    if dest.exists():
        shutil.rmtree(dest)

    roots = [p for p in (Path("/kaggle/input"), Path("/content"), HERE) if p.exists()]
    for root in roots:
        for zip_path in root.rglob("enzyme_classification*.zip"):
            print(f"extracting {zip_path}")
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(zip_path.parent)
            inner = zip_path.parent / "enzyme_classification"
            if (inner / "src" / "config.py").exists():
                if inner.resolve() != dest.resolve():
                    shutil.move(str(inner), str(dest))
                return dest

    if REPO_URL:
        print(f"cloning {REPO_URL}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        return dest

    raise SystemExit(
        "Could not find the project. Either\n"
        "  * zip the enzyme_classification folder (src/, config.py) and upload it as a "
        "Kaggle dataset / Colab file, or\n"
        "  * set ENZYME_REPO_URL to a git URL before running this cell."
    )

PROJECT = fetch_project(WORK)
sys.path.insert(0, str(PROJECT / "src"))

import runpy
def run_stage(script: str, *argv: str):
    """Execute src/<script> with the given argv, as if typed on the command line."""
    argv_backup = sys.argv
    sys.argv = [str(PROJECT / "src" / script), *argv]
    try:
        return runpy.run_path(str(PROJECT / "src" / script), run_name="__main__")
    finally:
        sys.argv = argv_backup

print("project:", PROJECT)
print("stages :", sorted(p.name for p in (PROJECT / "src").glob("*.py") if p.name[0].isdigit()))


## Run stage 03_model_rf

`python src/03_model_rf`


In [ ]:
run_stage("03_model_rf.py", '--skip-imbalance')


## Results


In [ ]:
# %% [markdown]
# Results
# Any stage that writes `artifacts/metrics/*.json` is summarised here.

import json, glob, os

METRICS = PROJECT / "artifacts" / "metrics"
if not METRICS.exists():
    print("no metrics yet")
else:
    rows = []
    for path in sorted(METRICS.glob("*.json")):
        try:
            payload = json.loads(path.read_text())
        except Exception:
            continue
        test = payload.get("test") or (payload.get("results") or {}).get("test") or {}
        if isinstance(test, dict) and "macro_f1" in test:
            rows.append({
                "run": path.stem,
                "representation": payload.get("representation") or payload.get("shared_representation", "-"),
                "val macro-F1": round(payload.get("validation", payload.get("results", {}).get("validation", {})).get("macro_f1", float("nan")), 4),
                "test macro-F1": round(test["macro_f1"], 4),
                "test MCC": round(test.get("mcc", float("nan")), 4),
                "test acc": round(test.get("accuracy", float("nan")), 4),
            })
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df if len(df) else "no scored runs found")

figs = sorted((PROJECT / "reports" / "figures").glob("*.png"))
if figs:
    print("\nfigures written:")
    for f in figs:
        print("  ", f.relative_to(PROJECT))
